In [6]:
import anywidget

import ipywidgets

import sys

print("Python:", sys.executable)

print("anywidget:", anywidget.__version__)

print("ipywidgets:", ipywidgets.__version__)

Python: /Applications/Blender.app/Contents/Resources/5.1/python/bin/python3.13
anywidget: 0.11.0
ipywidgets: 8.1.8


In [8]:
# %% 1) Make the gesture-recognizer-widget package importable in this kernel.
#        It (and its deps) must live in BLENDER's bundled Python, which is the
#        kernel running this notebook -- sys.executable points at it, so we
#        install into it once. To hack on a local checkout instead, swap the
#        package name below for "-e /path/to/caputre_motion" (see README).
import subprocess
import sys

try:
    import gesture_widget  # noqa: F401
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "gesture-recognizer-widget"]
    )

from gesture_widget import GestureRecognizerWidget  # noqa: E402

print("gesture_widget ready in Blender's Python")


# %% 2) Show the widget, then click "Start Camera" in the output and allow webcam
w = GestureRecognizerWidget()
w.sync_interval_ms = 33  # push landmarks to Python at ~30 Hz
w


gesture_widget ready in Blender's Python


In [10]:

# %% 3) A live hand armature, driven by bpy (main-thread safe).
#        One bone per joint connection; the timer sets each bone's head/tail
#        to the MediaPipe landmark positions to form a hand skeleton.
import bpy  # noqa: E402

SCALE = 15.0      # world units spanned by the camera frame (3x to match the
                  # larger box scene; bump this if your scene grows again)
BONE_REST = 0.4   # placeholder length for the bones at creation time;
                  # the timer overwrites head/tail with real positions.
SMOOTHING = 0.4   # EMA factor for de-jitter: 1.0 = raw (no smoothing),
                  # lower = smoother but laggier. ~0.3-0.5 is a good range.

# Traitlet names on `w` for all 21 landmarks of the MediaPipe hand-landmark
# model bundle. (Mapped to friendly object-name hints, though the bones get
# their positions straight from the traitlets.)
LANDMARKS = {
    "wrist": "HandLM_Wrist",
    "thumb_cmc": "HandLM_Thumb_CMC",
    "thumb_mcp": "HandLM_Thumb_MCP",
    "thumb_ip": "HandLM_Thumb_IP",
    "thumb_tip": "HandLM_Thumb_Tip",
    "index_finger_mcp": "HandLM_Index_MCP",
    "index_finger_pip": "HandLM_Index_PIP",
    "index_finger_dip": "HandLM_Index_DIP",
    "index_finger_tip": "HandLM_Index_Tip",
    "middle_finger_mcp": "HandLM_Middle_MCP",
    "middle_finger_pip": "HandLM_Middle_PIP",
    "middle_finger_dip": "HandLM_Middle_DIP",
    "middle_finger_tip": "HandLM_Middle_Tip",
    "ring_finger_mcp": "HandLM_Ring_MCP",
    "ring_finger_pip": "HandLM_Ring_PIP",
    "ring_finger_dip": "HandLM_Ring_DIP",
    "ring_finger_tip": "HandLM_Ring_Tip",
    "pinky_mcp": "HandLM_Pinky_MCP",
    "pinky_pip": "HandLM_Pinky_PIP",
    "pinky_dip": "HandLM_Pinky_DIP",
    "pinky_tip": "HandLM_Pinky_Tip",
}

# (parent, child) edges of the MediaPipe hand skeleton. Each edge is one bone:
# head at the parent landmark, tail at the child landmark.
CONNECTIONS = [
    ("wrist", "thumb_cmc"),
    ("thumb_cmc", "thumb_mcp"),
    ("thumb_mcp", "thumb_ip"),
    ("thumb_ip", "thumb_tip"),
    ("wrist", "index_finger_mcp"),
    ("index_finger_mcp", "index_finger_pip"),
    ("index_finger_pip", "index_finger_dip"),
    ("index_finger_dip", "index_finger_tip"),
    ("wrist", "middle_finger_mcp"),
    ("middle_finger_mcp", "middle_finger_pip"),
    ("middle_finger_pip", "middle_finger_dip"),
    ("middle_finger_dip", "middle_finger_tip"),
    ("wrist", "ring_finger_mcp"),
    ("ring_finger_mcp", "ring_finger_pip"),
    ("ring_finger_pip", "ring_finger_dip"),
    ("ring_finger_dip", "ring_finger_tip"),
    ("wrist", "pinky_mcp"),
    ("pinky_mcp", "pinky_pip"),
    ("pinky_pip", "pinky_dip"),
    ("pinky_dip", "pinky_tip"),
]


In [11]:
from mathutils import Vector  # noqa: E402

# Armature object holding one bone per connection.
ARM_NAME = "HandArmature"
arm_obj = bpy.data.objects.get(ARM_NAME)
if arm_obj is None:
    arm_obj = bpy.data.objects.new(ARM_NAME, bpy.data.armatures.new(ARM_NAME))
    bpy.context.scene.collection.objects.link(arm_obj)
arm_obj.show_in_front = False          # shaded octahedral look, like the rig
arm_obj.data.display_type = "OCTAHEDRAL"


def _bone_name(child_trait):
    # Each child landmark is the tail of exactly one bone, so it names it.
    return f"Bone_{child_trait}"


# The bone whose TAIL sits on landmark X is the parent of the bone whose HEAD
# sits on X -> this turns the flat edge list into proper finger chains.
_bone_ending_at = {child: _bone_name(child) for _p, child in CONNECTIONS}


# Create the bones once + parent them into finger chains. The timer drives the
# real head/tail each frame; Blender draws each as a proper octahedral bone
# whose width is auto-proportional to its length (no scaling involved).
bpy.context.view_layer.objects.active = arm_obj
bpy.ops.object.mode_set(mode="EDIT")
eb = arm_obj.data.edit_bones
for i, (_parent, child) in enumerate(CONNECTIONS):
    bone = eb.get(_bone_name(child)) or eb.new(_bone_name(child))
    bone.head = (i * BONE_REST, 0.0, 0.0)
    bone.tail = (i * BONE_REST, 0.0, BONE_REST)
for parent_lm, child in CONNECTIONS:
    bone = eb[_bone_name(child)]
    parent_name = _bone_ending_at.get(parent_lm)  # None for the wrist roots
    bone.parent = eb[parent_name] if parent_name else None
    bone.use_connect = False  # heads are driven explicitly each frame
bpy.ops.object.mode_set(mode="OBJECT")


def _img_to_world(p):
    # image space (x right, y DOWN) -> Blender (x right, z UP, y depth)
    x, y, z = p
    return Vector(((x - 0.5) * SCALE, z * SCALE, (0.5 - y) * SCALE))


# Per-landmark smoothing state (exponential moving average) to kill jitter.
_smoothed = {}


def _update_landmarks():
    # Read every landmark; bail until the whole hand is visible.
    pos = {}
    for trait in LANDMARKS:
        p = getattr(w, trait)  # [x, y, z] normalized 0..1, or [] when no hand
        if not p:
            return 0.033
        raw = _img_to_world(p)
        prev = _smoothed.get(trait)
        # EMA: ease the stored position toward the raw sample by SMOOTHING.
        pos[trait] = raw if prev is None else prev.lerp(raw, SMOOTHING)
        _smoothed[trait] = pos[trait]

    # Timers run on the main thread, so entering Edit mode and writing edit
    # bones here is safe. Setting head/tail directly = proper bones, no scaling.
    try:
        bpy.context.view_layer.objects.active = arm_obj
        bpy.ops.object.mode_set(mode="EDIT")
        eb = arm_obj.data.edit_bones
        for parent_lm, child in CONNECTIONS:
            head, tail = pos[parent_lm], pos[child]
            if (tail - head).length < 1e-4:  # avoid zero-length bones
                tail = head + Vector((0.0, 0.0, 1e-3))
            bone = eb[_bone_name(child)]
            bone.head = head
            bone.tail = tail
        bpy.ops.object.mode_set(mode="OBJECT")
    except RuntimeError:
        pass  # context not ready this tick; retry next frame
    return 0.033  # seconds until next call (~30 Hz). Return None to stop.


if bpy.app.timers.is_registered(_update_landmarks):
    bpy.app.timers.unregister(_update_landmarks)
bpy.app.timers.register(_update_landmarks)

print(f"Timer running — {len(CONNECTIONS)}-bone hand armature now tracks your hand.")


# %% 4) Stop driving Blender
# bpy.app.timers.unregister(_update_landmarks)
# w.stop()  # turn the camera off


Timer running — 20-bone hand armature now tracks your hand.


In [12]:
# %% 5) A cube on the index fingertip, used as a collider in the rigid-body sim.
#        The cube is a PASSIVE + kinematic (animated) rigid body: it's driven by
#        your fingertip each frame and shoves the scene's ACTIVE bodies (the box
#        tower in the screenshot) around as it moves through them.
#
#        Playback is YOURS: press Spacebar to play the timeline in real time.
#        The solver advances with the timeline; this timer only keeps the cube
#        glued to your fingertip. (It does NOT touch the current frame — doing so
#        fights Blender's playback and makes the frame ping-pong.)
import bpy  # noqa: E402
from mathutils import Vector  # noqa: E402

CUBE_NAME = "IndexFingerCollider"
CUBE_SIZE = 1.05          # edge length in world units (~3x, matches SCALE=15)
TIP_TRAIT = "index_finger_tip"

scene = bpy.context.scene

# A rigid-body world is required for any object to simulate. Create it once and
# give its point cache a long range so a long real-time playback won't run out.
if scene.rigidbody_world is None:
    bpy.ops.rigidbody.world_add()
rbw = scene.rigidbody_world
if rbw.collection is None:
    rbw.collection = bpy.data.collections.new("RigidBodyWorld")
rbw.point_cache.frame_start = scene.frame_start
rbw.point_cache.frame_end = 1_000_000
scene.frame_end = 1_000_000          # so playback doesn't loop back to the start

# Create the collider cube once and tag it as a kinematic passive rigid body.
cube = bpy.data.objects.get(CUBE_NAME)
if cube is None:
    bpy.ops.mesh.primitive_cube_add(size=CUBE_SIZE, location=(0.0, 0.0, 0.0))
    cube = bpy.context.active_object
    cube.name = CUBE_NAME
    bpy.context.view_layer.objects.active = cube
    bpy.ops.rigidbody.object_add()
    cube.rigid_body.type = "PASSIVE"
    cube.rigid_body.kinematic = True        # transform is driven by us, not solver
    cube.rigid_body.collision_shape = "BOX"

# Only track the fingertip. The timeline (Spacebar) advances frames and bakes
# the sim; on each played frame the solver reads the cube's live location.
_collider_state = {"pos": None}


def _drive_collider():
    p = getattr(w, TIP_TRAIT)  # [x, y, z] normalized, or [] when no hand
    if p:
        target = _img_to_world(p)
        prev = _collider_state["pos"]
        cube.location = target if prev is None else prev.lerp(target, SMOOTHING)
        _collider_state["pos"] = cube.location.copy()
    return 0.033                             # ~30 Hz; return None to stop


if bpy.app.timers.is_registered(_drive_collider):
    bpy.app.timers.unregister(_drive_collider)
bpy.app.timers.register(_drive_collider)

print(f"Collider cube '{CUBE_NAME}' tracks your fingertip. "
      "Press Spacebar to play the timeline, then move the box tower around.")


# %% 6) Stop driving the collider
# bpy.app.timers.unregister(_drive_collider)


Collider cube 'IndexFingerCollider' tracks your fingertip. Press Spacebar to play the timeline, then move the box tower around.
